# 기후 변화 피해국 EDA Tutorial
## Kaggle API로 실제 데이터를 받아서 분석하는 완전 가이드

**분석 목표:**
- 기후 변화로 가장 큰 피해를 입는 국가들의 공통된 특성 파악
- 경제적 취약성과 기후 피해의 상관관계 규명
- 자연재해 피해의 시계열 트렌드 분석

**사용 데이터셋 (Kaggle):**

| # | 데이터셋 | 원본 출처 | 내용 |
|---|----------|-----------|------|
| 1 | `brsdincer/all-natural-disasters-19002021-eosdis` | EM-DAT / CRED | 1900~2021 전세계 자연재해 이벤트 |
| 2 | `berkeleyearth/climate-change-earth-surface-temperature-data` | NASA / Berkeley Earth | 지표면 온도 이상치 시계열 |
| 3 | `kaggle/world-development-indicators` | World Bank | GDP, 인구, 농업 의존도 등 국가별 지표 |
| 4 | `thedevastator/nd-gain-country-index` | Notre Dame ND-GAIN | 기후 취약성 + 적응역량 국가 점수 |

**분석 흐름:**
```
[Kaggle API] → [Raw Data] → [전처리/병합] → [EDA 시각화] → [인사이트]
```

---
## Part 0 — 환경 설정 & Kaggle API 인증

### Step 0-1. 필요 패키지 설치

아래 셀을 처음 한 번만 실행하세요.

In [ ]:
%pip install -q kaggle pandas numpy matplotlib seaborn plotly geopandas

### Step 0-2. Kaggle API 키 설정

Kaggle API를 사용하려면 `kaggle.json` 파일이 필요합니다.

**발급 방법:**
1. [kaggle.com](https://www.kaggle.com) 로그인
2. 우측 상단 프로필 → **Settings**
3. **API** 섹션 → **Create New Token** 클릭
4. 다운로드된 `kaggle.json` 파일을 아래 경로에 복사:
   - **Windows:** `C:\Users\<사용자명>\.kaggle\kaggle.json`
   - **Mac/Linux:** `~/.kaggle/kaggle.json`

```json
// kaggle.json 내용 예시
{"username":"your_username","key":"xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"}
```

>  `kaggle.json`은 절대 GitHub에 올리면안된다. `.gitignore`에 추가

In [ ]:
import os
import json

# 너의 Local Computer 안에 있는 Path 경로를 확인할수 있는 Class
from pathlib import Path

# kaggle.json 위치 확인
kaggle_path = Path.home() / ".kaggle" / "kaggle.json"
print(kaggle_path)

if kaggle_path.exists():
    print(f"kaggle.json 발견: {kaggle_path}")
    with open(kaggle_path) as f:
        creds = json.load(f)
    print(f"사용자: {creds.get('username')}")
    # API키 보안: 앞 4자리만 표시
    key = creds.get('key')
    print(f"API Key: {key[:4]}{'*' * (len(key)-4)}")
else:
    print(f"kaggle.json 없음: {kaggle_path}")
    print("위의 Step 0-2 가이드를 따라 설정하세요.")

C:\Users\skcjf\.kaggle\kaggle.json
kaggle.json 발견: C:\Users\skcjf\.kaggle\kaggle.json
{'username': 'sjang1594', 'key': 'KGAT_ab9ac07098dbbc0ca8756eecb327e79c'}
사용자: sjang1594
API Key: KGAT*********************************
My Kaggle is : C:\Users\skcjf\.kaggle\kaggle.json


---
## Part 1 — 데이터 수집: Kaggle API

### Kaggle API 사용법 기초

```bash
# CLI로 직접 다운로드할 때
kaggle datasets download -d <owner>/<dataset-name> -p ./data --unzip

# 데이터셋 검색
kaggle datasets list -s "climate change"
```

Python에서는 `kaggle.api` 객체를 사용합니다.

### Step 1-1. 다운로드 경로 준비

In [12]:
import subprocess
from pathlib import Path
import zipfile

# Kaggle API로 데이터 다운로드
def download_kaggle_dataset(dataset_name, path):
    print(f"\n Downloading: {dataset_name}")
    print(f"saving to: {path}")
    
    cmd = [
        "kaggle",
        "datasets",
        "download",
        "-d",
        dataset_name,
        "-p",
        str(path)
    ]
    
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as e:
        print(f"Error downloading {dataset_name}: {e}")

# Data Unzipping
def unzip_in_place(path):
    for zip_file in path.glob("*.zip"):
        print(f"unzipping {zip_file.name}")
        
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall(path)

# 데이터 저장 폴더 생성
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# 각 데이터셋별 하위 폴더
dirs = {
    "disasters": DATA_DIR / "disasters",
    "temperature": DATA_DIR / "temperature",
    "wdi": DATA_DIR / "wdi",
    "ndgain": DATA_DIR / "ndgain",
}
    
### Kagggle dataset Mapping
datasets = {
    "disasters": "brsdincer/all-natural-disasters-19002021-eosdis",
    "temperature": "berkeleyearth/climate-change-earth-surface-temperature-data",
    "wdi": "kaggle/world-development-indicators",
    "ndgain": "thedevastator/nd-gain-country-index",
}

for name, path in dirs.items():
    path.mkdir(exist_ok=True)
    print(f"{path}")
    
    # download
    download_kaggle_dataset(datasets[name], path)
    
    # unzip
    unzip_in_place(path)
    
print("\n Data Download and Unzip Completed!")

data\disasters

 Downloading: brsdincer/all-natural-disasters-19002021-eosdis
saving to: data\disasters
unzipping all-natural-disasters-19002021-eosdis.zip
data\temperature

 Downloading: berkeleyearth/climate-change-earth-surface-temperature-data
saving to: data\temperature
unzipping climate-change-earth-surface-temperature-data.zip
data\wdi

 Downloading: kaggle/world-development-indicators
saving to: data\wdi
unzipping world-development-indicators.zip
data\ndgain

 Downloading: thedevastator/nd-gain-country-index
saving to: data\ndgain
Error downloading thedevastator/nd-gain-country-index: Command '['kaggle', 'datasets', 'download', '-d', 'thedevastator/nd-gain-country-index', '-p', 'data\\ndgain']' returned non-zero exit status 1.

 Data Download and Unzip Completed!
